# 第4章 数えるだけで漱石っぽくなる：bigramモデル

夏目漱石『こころ』で作る、はじめての言語モデル

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/04-bigram.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## 材料を手に入れる：青空文庫


In [ ]:
import urllib.request

url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/master/"
       "cards/000148/files/773_ruby_5968/773_ruby_5968.txt")
raw = urllib.request.urlopen(url).read()
print(type(raw), len(raw), "バイト")
print(raw[:40])

## 文字コードの壁を越える


In [ ]:
text = raw.decode("cp932")
print(text[:300])

## 掃除する：正規表現


In [ ]:
import re

sample = "私《わたくし》はその人を常に先生《せんせい》と呼んでいた。"
print(re.sub(r"《.*?》", "", sample))

In [ ]:
print(re.sub(r"《.*》", "", sample))

In [ ]:
def clean_aozora(text):
    text = text.split("-------------------------------------------------------")[2]  # 冒頭の説明を除く
    text = text.split("底本：")[0]                   # 末尾の書誌情報を除く
    text = re.sub(r"《.*?》", "", text)              # ルビを除く
    text = re.sub(r"※?［＃.*?］", "", text)          # 入力者注を除く
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)           # 3行以上の空行を2行に
    return text.strip()

text = clean_aozora(raw.decode("cp932"))
print(len(text), "文字")
print(text[:150])

In [ ]:
with open("kokoro.txt", "w", encoding="utf-8") as f:
    f.write(text)
print("保存しました")

## 文字を数える


In [ ]:
from collections import Counter

counts = Counter(text)
print("文字の種類：", len(counts))
print(counts.most_common(10))

## 「次の文字」を数える


In [ ]:
pairs = Counter(zip(text, text[1:]))
print(pairs.most_common(10))

In [ ]:
def next_counts(ch):
    c = Counter()
    for (a, b), n in pairs.items():   # 全部のペアの中から
        if a == ch:                   # 1文字目が ch のものだけ
            c[b] = n                  # 2文字目と回数を記録
    return c

c = next_counts("先")
print("「先」は全部で", sum(c.values()), "回")
print(c.most_common(5))

## 漱石に書かせる


In [ ]:
import random

following = {}
for a, b in zip(text, text[1:]):
    if a not in following:
        following[a] = Counter()
    following[a][b] += 1

def generate(start, n=80):
    out = start
    ch = start
    for _ in range(n):
        c = following[ch]
        ch = random.choices(list(c.keys()), weights=list(c.values()))[0]
        out += ch
    return out

random.seed(1)
print(generate("私"))

## なぜ1文字では足りないのか


In [ ]:
following3 = {}
for a, b, c in zip(text, text[1:], text[2:]):
    key = a + b
    if key not in following3:
        following3[key] = Counter()
    following3[key][c] += 1

def generate3(start, n=80):
    out = start
    for _ in range(n):
        c = following3.get(out[-2:])
        if c is None:
            break
        out += random.choices(list(c.keys()), weights=list(c.values()))[0]
    return out

random.seed(1)
print(generate3("私は"))
print("2文字の組み合わせの種類：", len(following3))

## 確率の表として眺める


In [ ]:
look = ["先", "生", "私", "は", "。"]
print("今＼次 " + " ".join(f"{b:>4}" for b in look))
for a in look:
    row = " ".join(f"{pairs[(a, b)]:>5}" for b in look)
    print(f"  {a}   {row}")